In [1]:
# CELL 1: Install latest libraries and clear broken cache
!pip install -U transformers huggingface_hub datasets -q
!rm -rf ~/.cache/huggingface
!rm -rf ./hf_cache
print("Libraries updated and cache cleared!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 141.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.5 MB/s eta 0:00:00
Libraries updated and cache cleared!


In [3]:
#Imports
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Data Pipeline -> Dataset from Github

url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url)

# Map 'positive' to 1 and 'negative' to 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split into train and test
train_df, test_df = train_test_split(df, train_size=10000, test_size=1000, random_state=42)

# Keep only the text column for training (simulating no labels)
train_df = train_df[['review']].rename(columns={'review': 'text'})
test_df = test_df[['review', 'label']].rename(columns={'review': 'text'})

print(f"Loaded {len(train_df)} training samples and {len(test_df)} test samples.")


Loaded 10000 training samples and 1000 test samples.


In [6]:
#Hueristic Rules and Weak Labelling

STRONG_POS = ["masterpiece", "excellent", "amazing", "perfect", "loved it", "brilliant", "best"]
WEAK_POS = ["good", "enjoyable", "decent", "fine", "liked", "entertaining"]
STRONG_NEG = ["terrible", "awful", "worst", "hate", "boring", "waste of time", "horrible"]
WEAK_NEG = ["bad", "poor", "disappointing", "slow", "dull"]

def apply_weak_labels(text):
    text_lower = str(text).lower()

    if any(word in text_lower for word in STRONG_POS): return 1, 1.0
    if any(word in text_lower for word in STRONG_NEG): return 0, 1.0

    if any(word in text_lower for word in WEAK_POS): return 1, 0.5
    if any(word in text_lower for word in WEAK_NEG): return 0, 0.5

    return -1, 0.0


weak_results = train_df['text'].apply(apply_weak_labels)
train_df['weak_label'] = weak_results.apply(lambda x: x[0])
train_df['confidence'] = weak_results.apply(lambda x: x[1])

# Filter out samples that didn't match any rules
train_df = train_df[train_df['weak_label'] != -1].reset_index(drop=True)
print(f"Training samples after weak labeling: {len(train_df)}")

Training samples after weak labeling: 8441


In [7]:
# PyTorch Dataset & Tokenizer (Hugging Face)


MODEL_NAME = "distilbert-base-uncased"
print(f"Downloading tokenizer for {MODEL_NAME}...")

# Force a fresh download to avoid any hidden cache issues
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, force_download=True)

class WeakSupervisionDataset(Dataset):
    def __init__(self, texts, labels, confidences, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.confidences = confidences
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'weights': torch.tensor(self.confidences[idx], dtype=torch.float)
        }

# Create datasets
train_dataset = WeakSupervisionDataset(
    train_df['text'].values,
    train_df['weak_label'].values,
    train_df['confidence'].values,
    tokenizer
)

# For test set, we use the TRUE labels to evaluate
test_dataset = WeakSupervisionDataset(
    test_df['text'].values,
    test_df['label'].values,
    np.ones(len(test_df)), # Test set uses true labels, so weight is 1.0
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

In [8]:
# Confidence-Weighted Loss Function

class ConfidenceWeightedLoss(nn.Module):
    def __init__(self):
        super(ConfidenceWeightedLoss, self).__init__()

    def forward(self, logits, targets, weights):
        # Calculate standard cross entropy loss without reduction
        loss = torch.nn.functional.cross_entropy(logits, targets, reduction='none')
        # Multiply loss by sample confidence weights and take the mean
        weighted_loss = loss * weights
        return weighted_loss.mean()

criterion = ConfidenceWeightedLoss()


In [9]:
#Training and Evaluation

def train_model(model, loader, use_weights=True):
    optimizer = AdamW(model.parameters(), lr=2e-5)
    model.train()

    for epoch in range(2): # 2 epochs for demonstration
        total_loss = 0
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            weights = batch['weights'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            if use_weights:
                loss = criterion(logits, labels, weights)
            else:
                loss = torch.nn.functional.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [10]:
# Execution: Baseline vs Robust Model

baseline_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, force_download=True).to(device)
train_model(baseline_model, train_loader, use_weights=False)
baseline_acc = evaluate_model(baseline_model, test_loader)
print(f"Baseline Test Accuracy: {baseline_acc*100:.2f}%\n")

robust_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, force_download=True).to(device)
train_model(robust_model, train_loader, use_weights=True)
robust_acc = evaluate_model(robust_model, test_loader)
print(f"Robust Test Accuracy: {robust_acc*100:.2f}%\n")

print("="*40)
print("FINAL SUMMARY:")
print(f"Baseline (Standard Loss): {baseline_acc*100:.2f}%")
print(f"Robust (Weighted Loss):  {robust_acc*100:.2f}%")
print(f"Improvement:             {(robust_acc - baseline_acc)*100:.2f}%")
print("="*40)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

...

Epoch 1 Loss: 0.2872


Epoch 2: 100%|██████████| 528/528 [03:45<00:00,  2.34it/s]


Epoch 2 Loss: 0.1660


Evaluating: 100%|██████████| 63/63 [00:09<00:00,  6.97it/s]

Robust Test Accuracy: 68.90%

FINAL SUMMARY:
Baseline (Standard Loss): 67.50%
Robust (Weighted Loss):  68.90%
Improvement:             1.40%
